# UMUD R93T — odzysk bazy R23A z kodu (Colab, GPU, bez treningu)

**Po co.** Decyzja 8.11: oba finały muszą przejść kontrolę odtwarzalności. W bazie B12 trzy części są dziś kopią starych wyników:
**PA bazy PNG** (R21), **FL bazy** (R17 refit) i **MT bazy** (R12C). Na Drive są modele i kod, które je wygenerowały. Ten notebook
liczy je od nowa z obrazów — **tym samym kodem R20** (sumy SHA-256 sprawdzane) i **tymi samymi wagami** — i porównuje z zapisanymi wynikami.

Co liczy (test 309 + OSF-35):

| część | jak | z czym porównuje |
|---|---|---|
| R20 | rozcięgna R12B (≥ 0,5) + pęczki R17 refit (bok 768, bez TTA, ≥ 0,65) → `r20_geometry.diagnose` | `test_detail.csv` R20 i maski R20 z `mask_cache` (piksel po pikselu) |
| PA wspieranych fragmentów | `pa_local_support_baseline` (źródło PA bazy PNG w R21) | baza R23A (55 obrazów) |
| FL bazy | `fl_mm` z R20 = FL R17 refit | baza R23A (303 obrazy) |
| MT R12B | oryginalna funkcja R12B (dwie największe składowe, `vertical_mean`) − stała z OSF-35 | baza R23A (309) |
| wariant „dwie składowe” | R20 na parze, którą wybiera MT R12B | dane do reguł dla 14 obrazów bez wsparcia |
| skala i bloki | czytnik linijki i bloki wideo z `umud_final` | rejestr skali R4A |

**Uruchomienie:** Colab → *Środowisko wykonawcze → Zmień typ* → GPU (L4 / A100 / T4) → **Uruchom wszystko**, zatwierdź dostęp do Drive.
Szacuję 10–20 minut. Postęp zapisuje się po każdym obrazie — po rozłączeniu uruchom ten sam plik ponownie.

**Wynik:** `UMUD_Challenge/outputs/R93T_v1_0_0/R93T_wyniki.zip` (kilka MB). **Prześlij ten plik w czacie.**

Potrzebne na Drive (wszystko już tam jest): `UMUD_Challenge/outputs/R17_v1_0_0/refit/best.pt`, `UMUD_R12B/best_dinov2_frozen.pt`,
obrazy testu (`UMUD_Challenge/data/competition_test_v1/`) i OSF-35 (`UMUD_Challenge/data/r12b_inputs/external_osf35/…`),
maski R20 (`UMUD_Challenge/outputs/R20_v1_0_0/mask_cache/`, opcjonalnie) i referencja OSF
(`UMUD_Challenge/data/r12b_inputs/nb10d_external_mt_reference.csv`, opcjonalnie).


In [ ]:
# 1. USTAWIENIA — zwykle nic nie zmieniasz.
RUN_NAME = 'R93T_v1_0_0'     # ten sam katalog = wznowienie
CHECK_IMAGE_SHA = True       # sprawdza, czy obrazy są dokładnie te z manifestów
RUN_OSF = True               # OSF-35: ponowne wyprowadzenie stałej MT R12C (1,3629 mm)
SESSION_HOURS = 3.0          # po tym czasie zapis i bezpieczne zatrzymanie
RUN_SELF_TEST = True


In [ ]:
# 2. ŚRODOWISKO I SAMOWYSTARCZALNY KOD (GPL-3.0-or-later). Nic tu nie trenuje.
import os, sys, time, json, hashlib, base64, zlib, subprocess, importlib.util, zipfile
from pathlib import Path
T_START = time.time()
FIXTURE = os.environ.get('R93T_FIXTURE_ROOT')   # tylko lokalne testy autora
if FIXTURE:
    DRIVE_ROOT = Path(FIXTURE) / 'MyDrive'; LOCAL = Path(FIXTURE) / 'content'; LOCAL.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive'); LOCAL = Path('/content')
PROJECT = DRIVE_ROOT / 'UMUD_Challenge'; RUN = PROJECT / 'outputs' / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
PACKAGE_SHA = '30a764f3f8aa0a13bbcc9d24e74443f2051ca4e25265b2da65b0360ca20a91e6'
PAYLOAD = '...'
raw = zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest() != PACKAGE_SHA:
    raise RuntimeError('Uszkodzony kod notebooka — wczytaj oryginalny plik ponownie')
SRC = LOCAL / ('r93t_src_' + PACKAGE_SHA[:12]); SRC.mkdir(parents=True, exist_ok=True)
for rel, content in json.loads(raw).items():
    path = SRC / rel
    if not path.resolve().is_relative_to(SRC.resolve()):
        raise ValueError('Invalid source path')
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(content)
for module, package in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('cv2', 'opencv-python-headless'),
                        ('skimage', 'scikit-image'), ('timm', 'timm'), ('PIL', 'pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Potrzebny torch. Wybierz standardowe środowisko GPU Colab.')
sys.path.insert(0, str(SRC))
for name in ['umud_core', 'umud_model', 'r20_geometry', 'r93_recover', 'umud_final', 'umud_final.scale', 'umud_final.layout']:
    sys.modules.pop(name, None)
import numpy as np, pandas as pd, torch, cv2
import umud_core
import r93_recover as R
CODE = R.verify_code(SRC)          # kod R20 bit w bit jak w R20 (inaczej stop)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu' and not FIXTURE:
    raise RuntimeError('Brak GPU. Colab: Środowisko wykonawcze → Zmień typ → GPU.')
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (testy)'
def log(*args):
    msg = ' '.join(str(a) for a in args); print(msg, flush=True)
    with (RUN / 'LOG.txt').open('a') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S ') + msg + '\n')
def show(frame, n=40):
    print(frame.head(n).to_string(index=False) if len(frame) else '(pusto)', flush=True)
code_dir = RUN / 'code'; code_dir.mkdir(exist_ok=True)
archive = code_dir / f'R93T_source_{PACKAGE_SHA[:12]}.zip'
if not archive.exists():
    tmp = archive.with_suffix('.tmp.zip')
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for rel, content in json.loads(raw).items():
            z.writestr(rel, content)
    tmp.replace(archive)
import timm, scipy, skimage
umud_core.atomic_json(RUN / 'ENVIRONMENT.json', {'python': sys.version, 'torch': torch.__version__, 'numpy': np.__version__,
    'pandas': pd.__version__, 'opencv': cv2.__version__, 'scipy': scipy.__version__, 'skimage': skimage.__version__,
    'timm': timm.__version__, 'gpu': GPU, 'cuda': torch.version.cuda, 'cudnn_benchmark': torch.backends.cudnn.benchmark,
    'cudnn_allow_tf32': torch.backends.cudnn.allow_tf32, 'matmul_allow_tf32': torch.backends.cuda.matmul.allow_tf32,
    'package_sha256': PACKAGE_SHA, 'r20_code_sha256': CODE})
ASSETS = SRC / 'assets'
DEADLINE = T_START + SESSION_HOURS * 3600
log('R93T |', GPU, '| kod', PACKAGE_SHA[:12], '| kod R20 zgodny z oryginałem |', RUN)


In [ ]:
# 3. WEJŚCIA: wagi (SHA-256), manifesty obrazów, wyniki referencyjne (R20, baza R23A, rejestr skali), prawda OSF.
if RUN_SELF_TEST:
    umud_core.atomic_json(RUN / 'SELF_TEST.json', R.selftest())
R17_SHA = os.environ.get('R93T_R17_SHA', R.R17_REFIT_SHA) if FIXTURE else R.R17_REFIT_SHA
R12B_SHA = os.environ.get('R93T_R12B_SHA', R.R12B_SHA) if FIXTURE else R.R12B_SHA
REG = R.registry(PROJECT, DRIVE_ROOT, R17_SHA, R12B_SHA)
umud_core.atomic_json(RUN / 'REGISTRY.json', REG)
test_csv = Path(os.environ.get('R93T_TEST_MANIFEST', ASSETS / 'test_manifest.csv')) if FIXTURE else ASSETS / 'test_manifest.csv'
osf_csv = Path(os.environ.get('R93T_OSF_MANIFEST', ASSETS / 'osf_manifest.csv')) if FIXTURE else ASSETS / 'osf_manifest.csv'
MAN = {'test': R.load_manifest(test_csv, PROJECT, CHECK_IMAGE_SHA)}
if RUN_OSF:
    MAN['osf'] = R.load_manifest(osf_csv, PROJECT, CHECK_IMAGE_SHA)
REF20 = pd.read_csv(Path(os.environ.get('R93T_REF20', ASSETS / 'r20_test_detail.csv')) if FIXTURE else ASSETS / 'r20_test_detail.csv')
BASE = pd.read_csv(Path(os.environ.get('R93T_BASE', ASSETS / 'base_r23a.csv')) if FIXTURE else ASSETS / 'base_r23a.csv')
SOURCES = pd.read_csv(Path(os.environ.get('R93T_SOURCES', ASSETS / 'r23a_sources.csv')) if FIXTURE else ASSETS / 'r23a_sources.csv')
SCALE_REG = pd.read_csv(ASSETS / 'scale_registry_309.csv')
CACHE_DIR = PROJECT / 'outputs' / 'R20_v1_0_0' / 'mask_cache'
R20_CACHE = {r.image_id: str(CACHE_DIR / f'{r.mask_key}.npz') for r in REF20.itertuples()} if CACHE_DIR.is_dir() else {}
N_CACHE = sum(Path(p).is_file() for p in R20_CACHE.values())
ref_osf = PROJECT / 'data' / 'r12b_inputs' / 'nb10d_external_mt_reference.csv'
if ref_osf.is_file():
    t = pd.read_csv(ref_osf)
    OSF_TRUTH = pd.DataFrame({'image_id': t['ImageID'].astype(str), 'mt_gt_mm': t['MT_gt_median'].astype(float)})
    OSF_TRUTH_SRC = 'nb10d_external_mt_reference.csv (MT_gt_median, dokładnie jak R12B)'
else:
    t = pd.read_csv(ASSETS / 'osf35_full.csv')
    OSF_TRUTH = pd.DataFrame({'image_id': t['ImageID'].astype(str),
                              'mt_gt_mm': t[[f'R{k}_MT' for k in range(1, 8)]].median(axis=1).astype(float)})
    OSF_TRUTH_SRC = 'mediana 7 ekspertów z osf35_full.csv (zastępczo — brak pliku referencji R12B)'
log('Wagi: R17 refit', REG['r17_refit']['sha256'][:12], '| R12B', REG['r12b']['sha256'][:12],
    '| obrazy: test', len(MAN['test']), '| OSF', len(MAN.get('osf', [])), '| maski R20 na Drive:', N_CACHE)
log('Prawda OSF:', OSF_TRUTH_SRC)


In [ ]:
# 4. PRZEBIEG — wznawialny po każdym obrazie (wyniki od razu na Drive).
BUDGET_HIT = False
OUT = RUN / 'export'
MODELS = R.load_models(REG, DEVICE)
log('R17 refit: config', MODELS['fasc_config'])
RECS = {}
try:
    RECS['test'] = R.run_dataset('test', MAN['test'], MODELS, REG, PACKAGE_SHA, DEVICE, OUT, DEADLINE, log, r20_cache=R20_CACHE)
    if RUN_OSF:
        RECS['osf'] = R.run_dataset('osf', MAN['osf'], MODELS, REG, PACKAGE_SHA, DEVICE, OUT, DEADLINE, log, map_ids=())
except R.SessionBudgetReached as e:
    BUDGET_HIT = True; log(str(e), '— uruchom wszystko ponownie w nowej sesji (ten sam RUN_NAME)')


In [ ]:
# 5. RAPORTY I PACZKA DO CZATU.
if not BUDGET_HIT:
    if RUN_OSF:
        T_OSF = R.tables(RECS['osf'], R.R12C_BIAS_MM)
        BIAS, N_BIAS, OSF_TAB = R.osf_bias(T_OSF['mt'], OSF_TRUTH)
        OSF_TAB.to_csv(OUT / 'osf_r12b_mt.csv', index=False); T_OSF['detail'].to_csv(OUT / 'osf_r20_detail.csv', index=False)
    else:
        BIAS, N_BIAS = R.R12C_BIAS_MM, 0
    T = R.tables(RECS['test'], BIAS)
    for k, v in T.items():
        v.to_csv(OUT / (f'test_{k}.csv.gz' if k.startswith('fragments') else f'test_{k}.csv'), index=False)
    CTRL = R.compare_r20(T['detail'], REF20); CTRL.to_csv(OUT / 'CONTROL_R20.csv', index=False)
    CB, CB_SUM = R.compare_base(T['detail'], T['mt'], BASE, SOURCES, BIAS); CB.to_csv(OUT / 'CONTROL_BASE_R23A.csv', index=False)
    _, CB_SUM_FIXED = R.compare_base(T['detail'], T['mt'], BASE, SOURCES, R.R12C_BIAS_MM)
    CACHE_SUM = R.cache_summary(T['cache'], REF20, BASE)
    SC = R.scale_and_blocks(MAN['test'], SCALE_REG, log); SC.to_csv(OUT / 'test_scale_blocks.csv', index=False)
    SUMMARY = {'status': 'R93T_COMPLETE', 'version': R.VERSION, 'gpu': GPU, 'package_sha256': PACKAGE_SHA,
               'rows': {k: len(v) for k, v in RECS.items()}, 'osf_truth': OSF_TRUTH_SRC,
               'r12c_bias_derived_mm': BIAS, 'r12c_bias_n': N_BIAS, 'r12c_bias_frozen_mm': R.R12C_BIAS_MM,
               'control_r20': CTRL.to_dict('records'), 'control_base': CB_SUM, 'control_base_fixed_bias': CB_SUM_FIXED,
               'r20_mask_cache': CACHE_SUM,
               'scale_equal_registry': int((SC.same_layout & (SC.abs_diff_px_per_cm < 1e-6)).sum()), 'scale_n': len(SC),
               'video_blocks': int(SC.block.max()) + 1 if (SC.block >= 0).any() else 0,
               'minutes': round((time.time() - T_START) / 60, 1), 'public_score': 'NOT_MEASURED'}
    SUMMARY = R.jsonable(SUMMARY); umud_core.atomic_json(OUT / 'SUMMARY.json', SUMMARY)
    ZIP = R.pack(OUT, RUN, extra=[RUN / 'LOG.txt', RUN / 'ENVIRONMENT.json', RUN / 'REGISTRY.json', RUN / 'SELF_TEST.json'])
    print('\nKONTROLA względem R20 test_detail (zgodne = |różnica| <= 1e-6 i to samo pokrycie):'); show(CTRL)
    print('\nMaski R20 z Drive:', json.dumps(CACHE_SUM, ensure_ascii=False))
    print('\nBaza R23A — PA (źródło R20): %s/%s dokładnie, max |Δ| %s' % (CB_SUM['pa_r20_exact_1e6'], CB_SUM['pa_r20_n'], CB_SUM['pa_r20_max_abs']))
    print('Baza R23A — FL: %s/%s dokładnie, max |Δ| %s' % (CB_SUM['fl_exact_1e6'], CB_SUM['fl_n'], CB_SUM['fl_max_abs']))
    print('Baza R23A — MT R12C (stała z OSF %.4f mm, n=%d): %s/%s w 1e-4 mm, %s w 0,01 mm; daleko: %s; brak: %s' % (
        BIAS, N_BIAS, CB_SUM['mt_exact_1e4'], CB_SUM['mt_n'], CB_SUM['mt_within_001'], CB_SUM['mt_far_ids'][:12], CB_SUM['mt_missing_ids'][:12]))
    print('Skala z linijki = rejestr: %d/%d | bloki wideo: %d' % (SUMMARY['scale_equal_registry'], len(SC), SUMMARY['video_blocks']))
    log('GOTOWE. Prześlij w czacie plik:', ZIP, f'({ZIP.stat().st_size / 1024:.0f} KB)')
else:
    log('Bieg zapisany. W nowej sesji: ten sam notebook → Uruchom wszystko.')


## Co dalej

- **`CONTROL_R20.csv`** — każdy pomiar R20 powinien się zgadzać co do 1e-6 (status, liczba fragmentów, `pa_local_support_baseline`,
  `fl_mm`). Jeśli maski różnią się o kilka pikseli (inna karta GPU), `SUMMARY.json → r20_mask_cache` pokaże, czy pomiary liczone
  na samych maskach R20 zgadzają się dokładnie — wtedy różnica to tylko arytmetyka GPU, nie kod.
- **`CONTROL_BASE_R23A.csv`** — PA bazy PNG (55 obrazów), FL bazy (303) i MT bazy (309) wobec R23A.
- **`test_detail_two.csv`, `test_fragments_two.csv.gz`, `maps/`** — dane do reguł dla 14 obrazów bez wsparcia (bez kolejnego biegu).
- **`test_scale_blocks.csv`** — skala z linijki i bloki wideo dla 309 obrazów (do finału bez rejestru i bez numerów ID).

Po przesłaniu `R93T_wyniki.zip` wpinam odtworzone tabele do notebooka Kaggle (finał odtwarzalny) i przygotowuję sondy kontrolne.
